In [7]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [8]:
# Load Census dataset
# First CSV row is a bogus index row (0,1,2,...); real headers are on row 2
census_path = "/Users/priyankaramachandran/Desktop/UCD Courses/CN Project/dataset/Census Data Sac.csv"

census = pd.read_csv(census_path, skiprows=1)

# Recover if the bogus index row was still used as headers
if str(census.columns[0]).isdigit():
    census.columns = census.iloc[0].astype(str).str.lower().str.strip()
    census = census.iloc[1:].reset_index(drop=True)
else:
    census.columns = census.columns.str.lower().str.strip()

census.head()

,name,b19013_001e,b01003_001e,b02001_002e,b02001_003e,b02001_005e,state,county,tract
0,Census Tract 1; Sacramento County; California,120694,3813,2834,75,203,6,67,100
1,Census Tract 2; Sacramento County; California,157109,3901,2938,38,235,6,67,200
2,Census Tract 3; Sacramento County; California,143774,3429,2565,99,142,6,67,300
3,Census Tract 4; Sacramento County; California,105531,3973,2470,57,335,6,67,400
4,Census Tract 5.01; Sacramento County; California,83523,1837,1034,130,140,6,67,501


In [9]:
census.shape

(363, 9)

In [10]:
# Column names were standardized during load
census.columns

Index(['name', 'b19013_001e', 'b01003_001e', 'b02001_002e', 'b02001_003e',
       'b02001_005e', 'state', 'county', 'tract'],
      dtype='object')

In [11]:
# Rename ACS variables to readable names
census = census.rename(columns={
    "name": "tract_name",
    "b19013_001e": "median_income",
    "b01003_001e": "total_population",
    "b02001_002e": "white_population",
    "b02001_003e": "black_population",
    "b02001_005e": "asian_population"
})

census.head()

,tract_name,median_income,total_population,white_population,black_population,asian_population,state,county,tract
0,Census Tract 1; Sacramento County; California,120694,3813,2834,75,203,6,67,100
1,Census Tract 2; Sacramento County; California,157109,3901,2938,38,235,6,67,200
2,Census Tract 3; Sacramento County; California,143774,3429,2565,99,142,6,67,300
3,Census Tract 4; Sacramento County; California,105531,3973,2470,57,335,6,67,400
4,Census Tract 5.01; Sacramento County; California,83523,1837,1034,130,140,6,67,501


In [12]:
# Convert numeric columns
numeric_cols = [
    "median_income",
    "total_population",
    "white_population",
    "black_population",
    "asian_population"
]

for col in numeric_cols:
    census[col] = pd.to_numeric(census[col], errors="coerce")

census.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   tract_name        363 non-null    object
 1   median_income     363 non-null    int64 
 2   total_population  363 non-null    int64 
 3   white_population  363 non-null    int64 
 4   black_population  363 non-null    int64 
 5   asian_population  363 non-null    int64 
 6   state             363 non-null    int64 
 7   county            363 non-null    int64 
 8   tract             363 non-null    int64 
dtypes: int64(8), object(1)
memory usage: 25.7+ KB


In [13]:
# Create full Census GEOID
census["state"] = census["state"].astype(str).str.zfill(2)
census["county"] = census["county"].astype(str).str.zfill(3)
census["tract"] = census["tract"].astype(str).str.zfill(6)

census["tract_geoid"] = (
    census["state"] +
    census["county"] +
    census["tract"]
)

census[["tract", "tract_geoid"]].head()

,tract,tract_geoid
0,000100,06067000100
1,000200,06067000200
2,000300,06067000300
3,000400,06067000400
4,000501,06067000501


In [14]:
# Remove missing values
before_rows = len(census)

census = census.dropna(subset=[
    "median_income",
    "total_population"
]).copy()

print("Rows before:", before_rows)
print("Rows after:", len(census))

Rows before: 363
Rows after: 363


In [15]:
census["white_pct"] = (
    census["white_population"] / census["total_population"]
)

census["black_pct"] = (
    census["black_population"] / census["total_population"]
)

census["asian_pct"] = (
    census["asian_population"] / census["total_population"]
)

census.head()

,tract_name,median_income,total_population,white_population,black_population,asian_population,state,county,tract,tract_geoid,white_pct,black_pct,asian_pct
0,Census Tract 1; Sacramento County; California,120694,3813,2834,75,203,06,067,000100,06067000100,0.743247,0.019670,0.053239
1,Census Tract 2; Sacramento County; California,157109,3901,2938,38,235,06,067,000200,06067000200,0.753140,0.009741,0.060241
2,Census Tract 3; Sacramento County; California,143774,3429,2565,99,142,06,067,000300,06067000300,0.748031,0.028871,0.041411
3,Census Tract 4; Sacramento County; California,105531,3973,2470,57,335,06,067,000400,06067000400,0.621696,0.014347,0.084319
4,Census Tract 5.01; Sacramento County; California,83523,1837,1034,130,140,06,067,000501,06067000501,0.562874,0.070768,0.076211


In [16]:
from pathlib import Path

output_dir = Path("../cleaned_dataset")
output_dir.mkdir(parents=True, exist_ok=True)

census.to_csv(output_dir / "census_sacramento_cleaned.csv", index=False)

print("Saved cleaned Census data to cleaned_dataset/census_sacramento_cleaned.csv")

Saved cleaned Census data to cleaned_dataset/census_sacramento_cleaned.csv
